# v2.2 Advanced XGBoost Training — Balanced Attack & Benign Detection

## Problem This Notebook Solves
- **v2.1 model**: Too many false positives — flagged YouTube/Netflix as ATTACK (72% weight on `bwd_packets_per_sec`)
- **First v2.2 attempt**: `scale_pos_weight=0.5` + `BETA=0.5` swung too far — attacks flagged as CLEAN

## This Version Targets BOTH:
- BENIGN Recall > 90% (no false alarms on real streaming traffic)
- ATTACK Recall > 90% (catches actual DNS attacks)

## Key Tuning Parameters
| Parameter | Value | Effect |
|---|---|---|
| `scale_pos_weight` | `0.8` | Slight BENIGN preference without killing attack detection |
| `BETA` | `2.0` | Threshold tuned to maximise ATTACK recall |
| `min_child_weight` | `5` | Allows finer splits for subtle attack patterns |
| Log-Transform | 8 features | Stops streaming traffic from looking like floods |

In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, fbeta_score
)
import warnings
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
plt.style.use('ggplot')
print('[OK] Libraries loaded')

## 1. Load Data

In [ ]:
# UPDATE THIS PATH to your latest balanced dataset
FILE_PATH = r"C:\Users\shenal\Downloads\reseraach\PCAPS_Used\Final_Balanced_Attack_and_Benign\Final_balanced_Attack_and_Benign_new_Shuffled.csv"

print(f'[INFO] Loading: {FILE_PATH}')
df = pd.read_csv(FILE_PATH)
print(f'[OK] Rows: {len(df):,}  Cols: {len(df.columns)}')
print(f'\nClass Distribution (before encoding):')
print(df['label'].value_counts())
df.head(3)

## 2. Preprocessing + Log-Transform

In [ ]:
# Drop identity columns
COLS_TO_DROP = ['src_ip', 'dst_ip', 'src_port', 'dst_port', 'protocol_number']
df = df.drop(columns=COLS_TO_DROP, errors='ignore')

# Fix NaN / Inf
df.replace([np.inf, -np.inf], 0, inplace=True)
df.fillna(0, inplace=True)

# Fixed label encoding: BENIGN=0, ATTACK=1
LABEL_MAP = {'BENIGN': 0, 'ATTACK': 1}
df['label'] = df['label'].str.upper().map(LABEL_MAP).fillna(0).astype(int)
print(f'Labels: {dict(df["label"].value_counts())}')

# Encode protocol
PROTOCOL_CLASSES = ['DOH', 'DOT', 'TRADITIONAL', 'UNKNOWN', 'TCP', 'UDP']
le_proto = LabelEncoder()
le_proto.fit(PROTOCOL_CLASSES)
df['protocol'] = df['protocol'].astype(str).apply(
    lambda x: x if x in PROTOCOL_CLASSES else 'UNKNOWN'
)
df['protocol'] = le_proto.transform(df['protocol'])

# KEY FIX: Log-transform highly skewed features
# Without this: Netflix/YouTube high traffic rates look identical to DDoS floods
# With this: the DIFFERENCE between 10 Mbps and 10,000 Mbps is compressed
#            so the model focuses on PATTERN not raw magnitude
LOG_FEATURES = [
    'bwd_packets_per_sec',      # Was 72% importance — MUST be compressed
    'flow_bytes_per_sec',
    'flow_packets_per_sec',
    'fwd_packets_per_sec',
    'dns_queries_per_second',
    'total_fwd_packets',
    'total_bwd_packets',
    'dns_amplification_factor',
]
print('\n[LOG-TRANSFORM] Compressing skewed features...')
for col in LOG_FEATURES:
    if col in df.columns:
        df[col] = np.log1p(df[col].clip(lower=0))
        print(f'  log1p({col})')

print('\n[OK] Preprocessing complete')

## 3. Train / Test Split

In [ ]:
X = df.drop(columns=['label'])
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
n_benign = (y_train == 0).sum()
n_attack = (y_train == 1).sum()
print(f'Train: BENIGN={n_benign:,}  ATTACK={n_attack:,}  ratio={n_attack/n_benign:.2f}')
print(f'Test : {X_test.shape}')

## 4. Advanced XGBoost Training

### Tuning Guide
If after retraining:
- **Attacks still missed** → raise `scale_pos_weight` toward `1.2`
- **Benign still flagged** → lower `scale_pos_weight` toward `0.6`

In [ ]:
# ============================================================
# PRIMARY TUNING KNOB
# scale_pos_weight = ATTACK importance relative to BENIGN
#   0.6 = fewer false alarms, may miss some attacks
#   0.8 = balanced starting point (recommended)
#   1.0 = equal weight (default XGBoost)
#   1.2 = more sensitive to attacks, more false alarms
# ============================================================
SCALE_POS_WEIGHT = 0.8

model = xgb.XGBClassifier(
    # Core
    objective          = 'binary:logistic',
    eval_metric        = ['logloss', 'auc'],
    use_label_encoder  = False,

    # Class balance
    scale_pos_weight   = SCALE_POS_WEIGHT,

    # Tree architecture
    n_estimators       = 800,
    learning_rate      = 0.03,
    max_depth          = 5,           # Prevents bwd_pps dominance
    min_child_weight   = 5,           # Allows finer attack splits (was 10 — too coarse)
    gamma              = 0.05,

    # Regularisation — forces model to use multiple features, not just bwd_pps
    subsample          = 0.8,
    colsample_bytree   = 0.7,
    colsample_bylevel  = 0.7,
    reg_alpha          = 0.05,
    reg_lambda         = 1.2,

    tree_method        = 'hist',
    random_state       = 42,
)

print(f'[TRAIN] scale_pos_weight={SCALE_POS_WEIGHT}, min_child_weight=5')
model.fit(
    X_train, y_train,
    eval_set = [(X_train, y_train), (X_test, y_test)],
    verbose  = 100,
)
print('[DONE] Training complete.')

## 5. Threshold Tuning for ATTACK Recall

**BETA=2.0** means: catching attacks is **4x more important** than avoiding false alarms.  
This will pick a threshold that maximises attack detection even at the cost of a few extra false alarms on benign.

In [ ]:
y_prob = model.predict_proba(X_test)[:, 1]

# beta=2.0 → optimise for attack RECALL
# beta=1.0 → balanced F1
# beta=0.5 → optimise for PRECISION (fewer false alarms)
BETA = 2.0

thresholds = np.arange(0.05, 0.95, 0.01)
best_thresh, best_score = 0.5, 0
results = []

for t in thresholds:
    y_pred_t = (y_prob >= t).astype(int)
    score = fbeta_score(y_test, y_pred_t, beta=BETA, zero_division=0)
    results.append((t, score))
    if score > best_score:
        best_score, best_thresh = score, t

print(f'[TUNING] Optimal threshold  = {best_thresh:.2f}  (F-{BETA} = {best_score:.4f})')
print(f'         Default threshold  = 0.50')

scores = [r[1] for r in results]
plt.figure(figsize=(10, 4))
plt.plot(thresholds, scores, color='steelblue')
plt.axvline(best_thresh, color='red', linestyle='--', label=f'Best={best_thresh:.2f}')
plt.axvline(0.5, color='gray', linestyle=':', label='Default=0.50')
plt.xlabel('Threshold'); plt.ylabel(f'F-beta (beta={BETA})')
plt.title('Threshold vs Classification Score'); plt.legend(); plt.show()

## 6. Balance Report — ATTACK Recall vs BENIGN Recall

In [ ]:
y_pred = (y_prob >= best_thresh).astype(int)

print(f'=== CLASSIFICATION REPORT (threshold={best_thresh:.2f}) ===')
print(classification_report(y_test, y_pred, target_names=['BENIGN','ATTACK'], digits=4))

auc = roc_auc_score(y_test, y_prob)
print(f'ROC-AUC: {auc:.4f}')

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['BENIGN','ATTACK'], yticklabels=['BENIGN','ATTACK'])
plt.title(f'Confusion Matrix @ threshold={best_thresh:.2f}')
plt.ylabel('Actual'); plt.xlabel('Predicted'); plt.tight_layout(); plt.show()

# Balance Report
tn, fp, fn, tp = cm.ravel()
benign_recall  = tn / (tn + fp) if (tn + fp) > 0 else 0
attack_recall  = tp / (tp + fn) if (tp + fn) > 0 else 0
false_pos_rate = fp / (fp + tn) if (fp + tn) > 0 else 0

print(f'\n===============================')
print(f' BALANCE REPORT')
print(f'===============================')
print(f' BENIGN Recall  : {benign_recall*100:6.1f}%  (% of real benign correctly identified)')
print(f' ATTACK Recall  : {attack_recall*100:6.1f}%  (% of real attacks caught)')
print(f' False Pos Rate : {false_pos_rate*100:6.1f}%  (% of benign wrongly flagged)')
print(f'===============================')

if attack_recall < 0.85:
    print('\n[ADVICE] Attack recall too low!')
    print('  -> Raise scale_pos_weight to 1.0 or 1.2 and retrain')
    print('  -> Or lower BETA threshold: try BETA=3.0 above')
elif benign_recall < 0.85:
    print('\n[ADVICE] Too many false alarms on benign!')
    print('  -> Lower scale_pos_weight to 0.6 and retrain')
    print('  -> Or raise BETA threshold: try BETA=1.0 above')
else:
    print('\n[GOOD] Both recalls above 85%. Model is well balanced!')

## 7. Feature Importance

In [ ]:
importance = pd.Series(model.feature_importances_, index=X.columns)
top20 = importance.nlargest(20)

plt.figure(figsize=(10, 7))
top20.sort_values().plot(kind='barh', color='steelblue')
plt.title('Top 20 Feature Importances (v2.2)')
plt.xlabel('Importance Score')
plt.tight_layout(); plt.show()

bwd_imp = importance.get('bwd_packets_per_sec', 0)
print(f'\n[CHECK] bwd_packets_per_sec importance = {bwd_imp:.4f}')
if bwd_imp > 0.30:
    print('  [WARN] Still dominant (>30%). Try reducing colsample_bytree to 0.5')
elif bwd_imp > 0.15:
    print('  [OK] Reduced but still significant. Acceptable.')
else:
    print('  [GOOD] Well distributed. Model uses many features now.')

## 8. Save Model Bundle

In [ ]:
bundle = {
    'model':            model,
    'threshold':        best_thresh,
    'log_features':     LOG_FEATURES,
    'protocol_classes': PROTOCOL_CLASSES,
    'scale_pos_weight': SCALE_POS_WEIGHT,
    'beta':             BETA,
}

with open('xgb_model_v2.2_advanced.pkl', 'wb') as f:
    pickle.dump(bundle, f)

print(f'[SAVED] xgb_model_v2.2_advanced.pkl')
print(f'  threshold        = {best_thresh:.2f}')
print(f'  scale_pos_weight = {SCALE_POS_WEIGHT}')
print(f'  beta             = {BETA}')
print(f'  ATTACK Recall    = {attack_recall*100:.1f}%')
print(f'  BENIGN Recall    = {benign_recall*100:.1f}%')